# Person + Head Detection -- 3 Tracks (Colab / Kaggle)

- **Track C** -- CNN detector train tu scratch (BatchNorm + Dropout), subset nho, muc dich hoc.
- **Track A** -- YOLO26 (Ultralytics), pretrained + fine-tune tren full 14k anh.
- **Track B** -- RF-DETR (transformer detector), pretrained + fine-tune tren full 14k anh.

**Truoc khi chay:**
- Colab: `Runtime > Change runtime type` -> chon GPU (T4).
- Kaggle: khung `Settings` ben phai -> `Accelerator` chon GPU T4 x2, va bat `Internet: On`
  (can internet de pip install + tai pretrained checkpoint).

Chay cell **SETUP** truoc tien, sau do chay 3 cell Track theo thu tu bat ky -- moi track doc lap,
khong phu thuoc lan nhau.


In [ ]:
# ============================================================
# SETUP -- chay cell nay truoc tien
# ============================================================
import os

# --------------------------------------------------------------
# 1) Chon nen tang dang chay: "colab" hoac "kaggle"
# --------------------------------------------------------------
PLATFORM = "colab"  # <-- doi thanh "kaggle" khi chay tren Kaggle

if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")

    # Sua duong dan nay tro dung toi file zip dataset trong Google Drive cua Andy
    # (upload "Person Detection with head.v2i.yolov8.zip" len Drive truoc)
    DATASET_ZIP = "/content/drive/MyDrive/Group01_ObjectDetection/Person Detection with head.v2i.yolov8.zip"
    DATASET_ROOT = "/content/dataset"
    OUTPUT_ROOT = "/content/outputs"

elif PLATFORM == "kaggle":
    # Tren Kaggle: bam "+ Add Input" -> upload cung file zip nay thanh 1 Kaggle Dataset truoc
    # (Kaggle tu giai nen khi upload dataset dang zip). Sua lai dung ten dataset da dat luc upload.
    DATASET_ZIP = None
    DATASET_ROOT = "/kaggle/input/person-detection-with-head"  # <-- sua dung ten dataset
    OUTPUT_ROOT = "/kaggle/working/outputs"

else:
    raise ValueError("PLATFORM phai la 'colab' hoac 'kaggle'")

os.makedirs(OUTPUT_ROOT, exist_ok=True)

if DATASET_ZIP is not None and not os.path.exists(os.path.join(DATASET_ROOT, "data.yaml")):
    import zipfile
    os.makedirs(DATASET_ROOT, exist_ok=True)
    print(f"Giai nen {DATASET_ZIP} -> {DATASET_ROOT} ...")
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(DATASET_ROOT)

assert os.path.exists(os.path.join(DATASET_ROOT, "data.yaml")), (
    f"Khong tim thay data.yaml trong {DATASET_ROOT} -- kiem tra lai DATASET_ZIP / DATASET_ROOT ben tren"
)

# --------------------------------------------------------------
# 2) Cai thu vien (torch + CUDA da co san tren ca Colab va Kaggle, khong can cai lai)
# --------------------------------------------------------------
!pip install -q ultralytics "rfdetr[train,loggers]" supervision

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("KHONG co GPU -- vao Runtime/Settings bat GPU accelerator roi Restart runtime")

CLASS_NAMES = ["Head", "Person"]
NUM_CLASSES = len(CLASS_NAMES)
DATA_YAML = os.path.join(DATASET_ROOT, "data.yaml")
TRAIN_IMAGES = os.path.join(DATASET_ROOT, "train", "images")
TRAIN_LABELS = os.path.join(DATASET_ROOT, "train", "labels")
VALID_IMAGES = os.path.join(DATASET_ROOT, "valid", "images")
VALID_LABELS = os.path.join(DATASET_ROOT, "valid", "labels")
TEST_IMAGES = os.path.join(DATASET_ROOT, "test", "images")
TEST_LABELS = os.path.join(DATASET_ROOT, "test", "labels")

print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## Track C -- CNN detector train tu scratch (BatchNorm + Dropout)

In [ ]:
# ============================================================
# TRACK C -- CNN detector train tu scratch (BatchNorm + Dropout)
# Doc lap voi Track A/B, chi can cell SETUP da chay truoc do.
# ============================================================
import random
from pathlib import Path

import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# ---------- Dataset ----------
def list_image_label_pairs(images_dir, labels_dir):
    images_dir, labels_dir = Path(images_dir), Path(labels_dir)
    pairs = []
    for img_path in sorted(images_dir.glob("*.jpg")):
        label_path = labels_dir / (img_path.stem + ".txt")
        if label_path.exists():
            pairs.append((img_path, label_path))
    return pairs


def read_yolo_labels(label_path):
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id, xc, yc, w, h = parts
            boxes.append((int(cls_id), float(xc), float(yc), float(w), float(h)))
    return boxes


class HeadPersonSubsetDataset(Dataset):
    """Single-box-per-cell grid dataset (YOLOv1-lite). Small subset on purpose --
    Track C exists to demonstrate training from scratch, not to be competitive."""

    def __init__(self, images_dir, labels_dir, subset_size=800, img_size=128, grid_size=8, seed=42):
        self.pairs = list_image_label_pairs(images_dir, labels_dir)
        random.Random(seed).shuffle(self.pairs)
        if subset_size is not None:
            self.pairs = self.pairs[:subset_size]
        self.img_size = img_size
        self.grid_size = grid_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, label_path = self.pairs[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        S = self.grid_size
        target = torch.zeros((5 + NUM_CLASSES, S, S), dtype=torch.float32)
        for cls_id, xc, yc, w, h in read_yolo_labels(label_path):
            gi = min(int(xc * S), S - 1)
            gj = min(int(yc * S), S - 1)
            if target[4, gj, gi] == 1.0:
                continue
            target[0, gj, gi] = xc * S - gi
            target[1, gj, gi] = yc * S - gj
            target[2, gj, gi] = w
            target[3, gj, gi] = h
            target[4, gj, gi] = 1.0
            target[5 + cls_id, gj, gi] = 1.0
        return img_t, target


# ---------- Model ----------
def conv_block(in_ch, out_ch, dropout=0.1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Dropout2d(dropout),
        nn.MaxPool2d(2),
    )


class ScratchDetector(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, grid_size=8, dropout=0.1):
        super().__init__()
        self.grid_size = grid_size
        self.backbone = nn.Sequential(
            conv_block(3, 16, dropout), conv_block(16, 32, dropout),
            conv_block(32, 64, dropout), conv_block(64, 128, dropout),
        )
        self.head = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(inplace=True), nn.Dropout2d(dropout),
            nn.Conv2d(128, 5 + num_classes, kernel_size=1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


# ---------- Loss (YOLOv1-style sum-squared-error) ----------
LAMBDA_COORD, LAMBDA_NOOBJ = 5.0, 0.5

class ScratchDetectorLoss(nn.Module):
    def forward(self, pred, target):
        pred_txty = torch.sigmoid(pred[:, 0:2]); pred_twth = torch.sigmoid(pred[:, 2:4])
        pred_obj = torch.sigmoid(pred[:, 4]); pred_cls = pred[:, 5:]
        target_txty, target_twth = target[:, 0:2], target[:, 2:4]
        target_obj, target_cls = target[:, 4], target[:, 5:]
        obj_mask, noobj_mask = target_obj, 1.0 - target_obj

        coord_loss = LAMBDA_COORD * (
            (obj_mask.unsqueeze(1) * (pred_txty - target_txty) ** 2).sum()
            + (obj_mask.unsqueeze(1) * (pred_twth - target_twth) ** 2).sum()
        )
        obj_loss = (obj_mask * (pred_obj - target_obj) ** 2).sum()
        noobj_loss = LAMBDA_NOOBJ * (noobj_mask * pred_obj ** 2).sum()
        cls_loss = (obj_mask.unsqueeze(1) * (pred_cls.sigmoid() - target_cls) ** 2).sum()
        return (coord_loss + obj_loss + noobj_loss + cls_loss) / pred.shape[0]


# ---------- Train ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Track C device:", device)

TRACK_C_OUT = Path(OUTPUT_ROOT) / "track_c"
TRACK_C_OUT.mkdir(parents=True, exist_ok=True)

SUBSET_SIZE, IMG_SIZE, GRID_SIZE, BATCH_SIZE, EPOCHS, LR = 800, 128, 8, 16, 20, 1e-3

train_ds = HeadPersonSubsetDataset(TRAIN_IMAGES, TRAIN_LABELS, SUBSET_SIZE, IMG_SIZE, GRID_SIZE, seed=42)
val_ds = HeadPersonSubsetDataset(VALID_IMAGES, VALID_LABELS, max(100, SUBSET_SIZE // 4), IMG_SIZE, GRID_SIZE, seed=123)
print(f"train: {len(train_ds)} images | val: {len(val_ds)} images")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = ScratchDetector(grid_size=GRID_SIZE).to(device)
criterion = ScratchDetectorLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_loss": [], "val_loss": []}
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), targets)
        loss.backward()
        optimizer.step()
        running += loss.item() * imgs.size(0)
    train_loss = running / len(train_ds)

    model.eval()
    running = 0.0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            running += criterion(model(imgs), targets).item() * imgs.size(0)
    val_loss = running / len(val_ds)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"epoch {epoch+1:>3}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

torch.save(model.state_dict(), TRACK_C_OUT / "scratch_detector.pt")
print("Track C: da luu model ->", TRACK_C_OUT / "scratch_detector.pt")

plt.figure(figsize=(6, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Track C - training curve")
plt.savefig(TRACK_C_OUT / "loss_curve.png")
plt.show()


## Track A -- YOLO26 (Ultralytics), pretrained + fine-tune

In [ ]:
# ============================================================
# TRACK A -- YOLO26 (Ultralytics), pretrained + fine-tune tren full dataset (14k anh)
# Doc lap voi Track C/B, chi can cell SETUP da chay truoc do.
# ============================================================
from pathlib import Path
from ultralytics import YOLO

TRACK_A_OUT = Path(OUTPUT_ROOT) / "track_a"

model_a = YOLO("yolo26n.pt")  # pretrained checkpoint tu dong tai lan dau

model_a.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,       # giam xuong 8 neu het VRAM (Colab/Kaggle free tier T4 ~16GB thuong on voi 16)
    project=str(TRACK_A_OUT),
    name="yolo26_finetune",
)

metrics_a = model_a.val(data=DATA_YAML, split="test")
print("Track A mAP50:   ", metrics_a.box.map50)
print("Track A mAP50-95:", metrics_a.box.map)


## Track B -- RF-DETR (transformer detector), pretrained + fine-tune

In [ ]:
# ============================================================
# TRACK B -- RF-DETR (transformer detector), pretrained + fine-tune tren full dataset (14k anh)
# Doc thang dataset dang YOLO co san (data.yaml + train/valid + images/labels),
# khong can convert sang COCO. Doc lap voi Track C/A, chi can cell SETUP da chay truoc do.
# ============================================================
from pathlib import Path
import numpy as np
from PIL import Image
import supervision as sv
from supervision.metrics import MeanAveragePrecision
from rfdetr import RFDETRNano

def list_image_label_pairs(images_dir, labels_dir):
    images_dir, labels_dir = Path(images_dir), Path(labels_dir)
    pairs = []
    for img_path in sorted(images_dir.glob("*.jpg")):
        label_path = labels_dir / (img_path.stem + ".txt")
        if label_path.exists():
            pairs.append((img_path, label_path))
    return pairs


def read_yolo_labels(label_path):
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id, xc, yc, w, h = parts
            boxes.append((int(cls_id), float(xc), float(yc), float(w), float(h)))
    return boxes


TRACK_B_OUT = Path(OUTPUT_ROOT) / "track_b"

model_b = RFDETRNano()  # pretrained checkpoint tu dong tai lan dau

model_b.train(
    dataset_dir=DATASET_ROOT,
    epochs=50,
    batch_size=8,        # Colab/Kaggle T4 (~16GB) chiu duoc batch lon hon RTX 2060 local (4)
    lr=1e-4,
    output_dir=str(TRACK_B_OUT / "rfdetr_finetune"),
)

# ---------- Quick mAP eval on the test split ----------
def ground_truth_detections(label_path, img_w, img_h):
    boxes, class_ids = [], []
    for cls_id, xc, yc, w, h in read_yolo_labels(label_path):
        x1, y1 = (xc - w / 2) * img_w, (yc - h / 2) * img_h
        x2, y2 = (xc + w / 2) * img_w, (yc + h / 2) * img_h
        boxes.append([x1, y1, x2, y2]); class_ids.append(cls_id)
    if not boxes:
        return sv.Detections.empty()
    return sv.Detections(xyxy=np.array(boxes, dtype=np.float32), class_id=np.array(class_ids, dtype=int))

best_ckpt = TRACK_B_OUT / "rfdetr_finetune" / "checkpoint_best_total.pth"
model_b_eval = RFDETRNano.from_checkpoint(str(best_ckpt))

test_pairs = list_image_label_pairs(TEST_IMAGES, TEST_LABELS)[:200]  # cap for a quick eval pass
metric = MeanAveragePrecision()
predictions, targets = [], []
for img_path, label_path in test_pairs:
    image = Image.open(img_path).convert("RGB")
    preds = model_b_eval.predict(image, threshold=0.3)
    predictions.append(preds)
    targets.append(ground_truth_detections(label_path, image.width, image.height))

result = metric.update(predictions, targets).compute()
print("Track B mAP50:   ", result.map50)
print("Track B mAP50-95:", result.map50_95)
